# Dataset main
---
### This notebook will work as the main workspace for the dataset that I ll work on my thesis. 

Αρχικά ας φτιάξω μια μορφή ευκόλη για μελετή του dataset. Θα φτιάξω ένα αντίγραφω της όπου έναντι του να είναι σε txt file θα το φτιάξω σε φορμάτ csv. ΔΕΝ υπάρχει ούτε σκοπός ούτε λόγος να λειτουργίσω με αυτό το αρχείο στο project παρά μόνο θα δημιουργηθεί για να μπορέσω να τα διαβάσω με καλύτερη άνεση.

In [ ]:
import pandas as pd 
import os

#because the dataset is untitled we have to specify the column names
# Η πλήρης λίστα με τα ονόματα των στηλών
col_names = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins',
    'logged_in', 'num_compromised', 'root_shell', 'su_attempted',
    'num_root', 'num_file_creations', 'num_shells', 'num_access_files',
    'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count',
    'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate',
    'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate',
    'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate',
    'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
    'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty_level'
]

df = pd.read_csv('dataset/KDDTrain+_20Percent.txt',sep=',', names=col_names)
print(df.shape)
print(df.head())

file_path = 'dataset/KDDTrain+_20Percent.csv'

if not os.path.exists(file_path):
    df.to_csv(file_path, index=False)
    print(f"Dataset saved to {file_path}")
else:    
    print(f"File {file_path} already exists. Skipping save.")


#### Dataset split 
---
αρχικά για να μπορέσω να δουλέψω θα χρειαστώ να σπάσω το dataset σε set διδασκαλίας και set test. Θα χρειαστή να βάλω σε κάθε στήλη ένα τύπο βαρήτητας για να μπορέσω να κάνω τον διαχωρισμό δίκαιο και περιεκτικό.

#### Data normalization 
----
Πρέπει να φτοιάξω τα δεδομένα μου με τρόπο όπου να μπορέσω να μετατρέψω την ουσία της πληροφορίας με τρόπο που να αποτελεί συγκρίσημη ποσότητα με τις υπόλοιπες στήλες ώστε να μπορέσω να συνχωνέψω τα δεδομένα και στην πορεία να τα ψηφιοποιήσω σε ένα simulation όπου να μπορεί να τρέψει σε πραγματικό χρόνο.

#### src_byte_cat
---
Για να χωρίσω δίκαια το dataset θα το κάνω με γνώμονα δύο μετρικές, αρχικά και βασικότερο το label της κύνησης καθώς παρέχει την πιο ποιοτικά σωστή πληροφορία για την ανάλυση που θέλουμε να κάνουμε. Δεύτερο ρόλο (αν και δεν έχει ερευνηθεί αρκετά) θα πάρει το μέγεθος του πακέτου. Επείδη όμως ένα πακέτο σπανίως έχει τέλειο και σταθερό μέγεθος αλλά και επειδή το ακριβές ποσό δεν μας δίνει κάποια ουσιόδη πληροφορία ο διαχωρισμός θα γίνει βάση της κατηγορίας που ανοίκει. Δηλαδή αν είναι από τα πιο ελαφρία, μεσαία ή βαριά πακέτα

Ακουλουθούν δύο μπλοκ κώδικα:
1. θα δημιουργήσει την καινούργια αυτή μετρική
2. Θα κάνει τον διαχωρισμό των δεδομένων σε test and training datase

In [ ]:
df['src_bytes_cat'] = pd.qcut(df['src_bytes'], q=4, duplicates='drop', labels=['low', 'medium', 'high'])
print(df['src_bytes_cat'].value_counts(normalize=True))
check_bins = df.groupby('src_bytes_cat')['src_bytes'].agg(['min', 'max', 'count'])
print(check_bins)

In [ ]:
df['strat_col'] = df['label'].astype(str) + "_" + df['src_bytes_cat'].astype(str)

counts = df['strat_col'].value_counts()
problematic_combinations = counts[counts < 2].index

print(f"Αρχικά προβληματικοί συνδυασμοί: {len(problematic_combinations)}")


mask = df['strat_col'].isin(problematic_combinations)
df.loc[mask, 'strat_col'] = df.loc[mask, 'label']

# Βήμα 4: Τελικός Έλεγχος
final_counts = df['strat_col'].value_counts()
remaining_singles = final_counts[final_counts < 2]

print(f"Εναπομείνασες μοναχικές εγγραφές: {len(remaining_singles)}")

# Αν υπάρχουν ακόμα singles, δες ποιες είναι:
if len(remaining_singles) > 0:
    print("\nΠΡΟΣΟΧΗ: Αυτές οι επιθέσεις υπάρχουν ΜΟΝΟ 1 φορά σε όλο το dataset:")
    print(remaining_singles)

τις Μοναδικές αυτές επιθέσεις θα τις απομονώσω και θα τις βάλω χειροκίνητα στο trainingset καθώς θα μπορέσω να βάλω κάθε άλλη εγγραφή με λίγες γραμμές κώδικα

In [ ]:
from sklearn.model_selection import train_test_split
#Βαση της μασκας διαχωρίζω τις μοναχικές εγγραφές από τις υπόλοιπες
singelMaks= df['strat_col'].isin(remaining_singles.index)
dfSingles = df[singelMaks] #μοναχικές εγγραφές
dfNonSingles = df[~singelMaks] #υπόλοιπες εγγραφές

train_majority, test_set = train_test_split(
    dfNonSingles,
    test_size=0.2, 
    random_state=42, 
    stratify=dfNonSingles['strat_col']
)
train_set = pd.concat([train_majority, dfSingles])
test_set = test_set.drop(columns=['strat_col'])
train_set = train_set.drop(columns=['strat_col'])
df = df.drop(columns=['strat_col'])
print(f"Μέγεθος του τελικού training set: {train_set.shape}")
print(f"Μέγεθος του τελικού test set: {test_set.shape}")

In [ ]:
xTrain = train_set.drop(columns=['label', 'difficulty_level'])
lablesTrain = train_set['label']

xTest = test_set.drop(columns=['label', 'difficulty_level'])
labelsTest = test_set['label']

---
#### Encoder 
Χρησιμοποιούμε Target Encoding. Με αυτή την τεχνική, κάθε κατηγορικό δεδομένο (π.χ. TCP) αντικαθίσταται από έναν αριθμό που αντιπροσωπεύει τον μέσο όρο του στόχου (target) για τη συγκεκριμένη κατηγορία στο σύνολο εκπαίδευσης.

Αν το TCP πάρει την τιμή 0.7, αυτό σημαίνει ότι το 70% των TCP συνδέσεων στο Train set ήταν επιθέσεις. Αν το UDP πάρει την τιμή 0.2, σημαίνει ότι μόνο το 20% των UDP ήταν επιθέσεις.

Έτσι, η σύγκριση 0.7 > 0.2 αποκτά ουσιαστικό νόημα για την ασφάλεια: Το μοντέλο καταλαβαίνει άμεσα ότι το TCP είναι στατιστικά πιο 'επικίνδυνο' από το UDP, ανεξάρτητα από το πόσο συχνά εμφανίζεται στη βάση.

In [ ]:
from sklearn.preprocessing import TargetEncoder
import numpy as np

categorical_cols = ['protocol_type', 'service', 'flag', 'src_bytes_cat']

yTrainBinary = np.where(lablesTrain == 'normal.', 0, 1)
encoder = TargetEncoder(target_type='binary', smooth="auto")
xTrain_encoded = encoder.fit_transform(xTrain[categorical_cols], yTrainBinary)
xTest_encoded = encoder.transform(xTest[categorical_cols])

df_encoded_train = pd.DataFrame(xTrain_encoded, columns=categorical_cols, index=xTrain.index)
df_encoded_test = pd.DataFrame(xTest_encoded, columns=categorical_cols, index=xTest.index)

xTrain_final = pd.concat([xTrain.drop(columns=categorical_cols), df_encoded_train], axis=1)
xTest_final = pd.concat([xTest.drop(columns=categorical_cols), df_encoded_test], axis=1)


print(f"Τελικό training set shape: {xTrain_final.shape}")
print(f"Τελικό test set shape: {xTest_final.shape}")


---
#### Κανονικοποίηση 
Σε αυτό το κομμάτι κανονικοποιηούμε τις στήλες του DF μέσο της τεχνικής StandardScale για να μπορέσουμε να έχουμε αριθμιτικές αξίες δουλέψημες σε τεχνική PCA

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
xTrain_scaled = scaler.fit_transform(xTrain_final)
xTest_scaled = scaler.transform(xTest_final)

---
#### PCA 
Εδώ θα εφαρμόσουμε την τεχνική του PCA για να συμπτίξουμε όσο μπορούμε τον τελικό πίνακα χωρίς να χάσουμε πληροφόρια. Το επιθυμητό αποτέλεσμα είναι να φτάσω σε ένα σημείο όπου να έχω πληροφορία ευκολός κωδικοποιήσημη και ευκολοδούλευτη μέσο του qiskit

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=16)

xTrain_pca = pca.fit_transform(xTrain_scaled)
xTest_pca = pca.transform(xTest_scaled)

# Πόσες στήλες μας έμειναν;
print(f"Explained Variance: {pca.explained_variance_ratio_.sum():.2%}")
print(f"Στόχος επετεύχθη: 16 στήλες για 4 Qubits (Amplitude) ή 16 Qubits (Angle).")